# Unified AI API — routing, policy and payload evidence

Validate OpenAI, inference, Responses and optionally Gemini paths; authenticated
discovery, valid/missing/invalid keys, cross-product isolation, debug headers, path
rejection, model RBAC and complete SSE streaming. Models come from the selected
environment's catalog plus explicit capabilities, never runtime name defaults.

Create or update a stable, least-privilege access contract through an editable .bicepparam.
Additional stress sampling is opt-in. Cleanup removes only this run's product.
Run the environment setup notebook first; execute sequentially in a fresh kernel.

## 1. Editable configuration

Disabled integrations are explicit SKIP; enabled but unavailable operations are BLOCKED.

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


Per-surface overrides optionally pin an actual deployed model, for example
`responses_model_override = "my-responses-model"` or
`gemini_model_override = "my-gemini-model"`. Leave overrides `None` for selection.
`test_gemini = True` requires that provider's backend. Additional stress is opt-in:
`enable_stress = True; stress_requests = 20`.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Editable configuration', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    from urllib.parse import quote
    from requests.structures import CaseInsensitiveDict
    from validationtools import ValidationContext, ValidationRun, ModelCatalog, RequestTransport, ValidationBlocked
    from apimtools import APIMClientTool
    from contracttools import scoped_deployment_name
    from contracttools import access_policy, prepare_artifact, deploy_contract
    from consumerhelpers import (
        access_artifact, contract_identifiers, cli_json, chat_content, error_code,
        complete_backend_parameters, load_contract_manifest, delete_owned_product, product_exists,
        review_backend_artifact,
    )
    environment = None
    from_azd = True
    overrides = {}  # Explicit canonical config/capabilities override the selected azd environment.
    run_id = time.strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    cleanup_enabled = False

    openai_model_override = None
    openai_backend_type = "ai-foundry"  # Set azure-openai for a dedicated Azure OpenAI backend.
    inference_model_override = None
    responses_model_override = None
    streaming_model_override = None
    gemini_model_override = None
    test_responses_api = True
    test_gemini = False
    enable_stress = False
    stress_requests = 20  # 1..100 sequential requests; no invisible retries
    run = ValidationRun("unified-ai")
    run_dir = root / "validation" / ".runs" / "unified-ai" / run_id

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. Context, capability-aware selection and exact API discovery

Annotate chat, responses and streaming independently. OpenAI deployment-path coverage requires OpenAI model-format metadata or explicit openai-deployments capability.

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. Context, capability-aware selection and exact API discovery', inputs={'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd'), 'openai_model_override': globals().get('openai_model_override'), 'inference_model_override': globals().get('inference_model_override'), 'responses_model_override': globals().get('responses_model_override')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    utils.require_inputs(mandatory_inputs)
    utils.require_capabilities(context.catalog)
    transport = RequestTransport(timeout=60)
    utils.format_output('2. Context, capability-aware selection and exact API discovery', {'value_1': "Chat-capable configured models:", 'value_2': context.catalog.models_for("chat")}, status='info', part='RESULT')

    selected = {}
    for label, operation, backend_type, override, enabled in [
        ("openai", "chat", "ai-foundry", openai_model_override, True),
        ("inference", "chat", "ai-foundry", inference_model_override, True),
        ("responses", "responses", "ai-foundry", responses_model_override, test_responses_api),
        ("streaming", "streaming", "ai-foundry", streaming_model_override, True),
        ("gemini", "chat", "gemini-openai", gemini_model_override, test_gemini),
    ]:
        if not enabled:
            run.skip(f"{label}/selection", "Disabled explicitly in configuration.")
            continue
        try:
            if label == "openai":
                candidates = [entry.name for entry in context.catalog.entries
                              if entry.backend_type == openai_backend_type and entry.name in context.catalog.models_for("chat")
                              and (entry.model_format == "OpenAI" or "openai-deployments" in entry.capabilities)]
                requested = override or next(iter(candidates), None)
                if requested not in candidates:
                    raise ValueError("OpenAI deployment path needs OpenAI format metadata or explicit openai-deployments capability.")
                selected[label] = context.catalog.select(operation, override=requested, backend_type=openai_backend_type)
            else:
                selected[label] = context.catalog.select(operation, override=override, backend_type=backend_type)
        except (ValidationBlocked, ValueError) as exc:
            if override is not None or label in ("openai", "inference"):
                run.blocked(f"{label}/selection", str(exc))
                raise
            run.skip(f"{label}/selection", f"No eligible model for this operation: {exc}")
    if "openai" not in selected or "inference" not in selected:
        raise RuntimeError("Required OpenAI and inference model selections are blocked.")
    utils.format_output('2. Context, capability-aware selection and exact API discovery', {'value_1': "Selected models:", 'selected': selected}, status='info', part='RESULT')


    apim = APIMClientTool(context.config["resource_group"], context.config.get("apim_name") or "")
    apim.initialize_context(context.account, enumerate_subscriptions=False)
    utils.format_output('2. Context, capability-aware selection and exact API discovery', {'value_1': "Environment:", 'value_2': environment or "(active)", 'value_3': "APIM:", 'value_4': apim.apim_resource_name}, status='info', part='RESULT')
    unified_api = apim.discover_api_exact("unified-ai")
    base_url = unified_api["url"]

    run_dir.mkdir(parents=True, exist_ok=True)


## 3. Configure, generate and preview the access contract

The visible policy contains model RBAC, capacity, and debug-header enablement. API grant is Unified AI only.

Prepare the 3. Configure, generate and preview the access contract state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_dir`, `run_id`, `selected`

**Output:** `allowed_models`, `use_case`, `product_id`, `policy_xml`, `artifact` are available to the following cells.


In [ ]:
with utils.notebook_step('3. Configure, generate and preview the access contract', inputs={'run_dir': globals().get('run_dir'), 'run_id': globals().get('run_id'), 'selected': globals().get('selected')}, expected='`allowed_models`, `use_case`, `product_id`, `policy_xml`, `artifact` are available to the following cells.', namespace=globals()):
    allowed_models = list(dict.fromkeys(selected.values()))
    use_case = {"businessUnit": "Validation", "useCaseName": "Unified", "environment": "DEV"}
    product_id = f"LLM-{use_case['businessUnit']}-{use_case['useCaseName']}-{use_case['environment']}"
    policy_xml = access_policy(allowed_models, 10000, 100000, "Monthly",
                              inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
    utils.format_output('3. Configure, generate and preview the access contract', {'policy_xml': policy_xml}, status='info', part='RESULT')
    artifact = access_artifact(root, run_dir, context, apim, use_case, ["unified-ai-api"], policy_xml)
    utils.format_output('3. Configure, generate and preview the access contract', {'value_1': artifact.preview()}, status='info', part='RESULT')
    utils.format_output('3. Configure, generate and preview the access contract', {'value_1': "Edit before deploying:", 'value_2': artifact.params_file}, status='info', part='RESULT')


## 4. Deploy the actual edited .bicepparam and get only its subscription key

Deploy the reviewed artifact for 4. Deploy the actual edited .bicepparam and get only its subscription key, then inspect its real resource outputs and ownership.

**Input:** `product_id`, `artifact`, `base_url`, `openai_api_version`, `inference_api_version`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4. Deploy the actual edited .bicepparam and get only its subscription key', inputs={'product_id': globals().get('product_id'), 'artifact': globals().get('artifact'), 'base_url': globals().get('base_url'), 'openai_api_version': globals().get('openai_api_version'), 'inference_api_version': globals().get('inference_api_version')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    product_preexists = product_exists(context, apim, product_id)
    outputs = deploy_contract(artifact.params_file, context.config["location"], scoped_deployment_name("validation-unified-access", apim.apim_service_id))
    product_id, subscription_name = contract_identifiers(outputs, product_id)
    api_key = apim.get_subscription_key(subscription_name)
    artifact.record_resource(f"{apim.apim_service_id}/products/{product_id}", created=not product_preexists)
    headers = {"api-key": api_key}
    openai_url = f"{base_url}/openai/deployments/{quote(selected['openai'], safe='')}/chat/completions?api-version={openai_api_version}"
    inference_url = f"{base_url}/models/chat/completions?api-version={inference_api_version}"
    responses_url = f"{base_url}/openai/responses"
    gemini_url = f"{base_url}/v1beta/openai/chat/completions"
    deployments_url = f"{base_url}/deployments"
    messages = [{"role": "user", "content": "What is 2+2? Answer briefly."}]

    def record(case, result, expected_status=200, body_check=None, **evidence):
        result.headers = CaseInsensitiveDict(result.headers)
        valid_body = True
        if body_check is not None:
            try:
                valid_body = bool(body_check(result.json_body))
            except (TypeError, ValueError, KeyError, IndexError, AttributeError):
                valid_body = False
        run.check(
            case, not result.error and result.status_code == expected_status and valid_body,
            expected=f"HTTP {expected_status}; expected response shape/policy evidence",
            actual=result.error or str(result.status_code),
            duration=result.duration,
            headers={k: v for k, v in result.headers.items() if k.lower().startswith("uaig-")},
            **evidence,
        )
        return result


## 5. Authenticated discovery, exact deployment and deliberate missing name

Reconcile configured allowed models with the live response; configuration alone is not proof of deployment.

Exercise 5. Authenticated discovery, exact deployment and deliberate missing name and compare the actual response/evidence with the expected feature behavior.

**Input:** `deployments_url`, `headers`, `allowed_models`, `selected`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('5. Authenticated discovery, exact deployment and deliberate missing name', inputs={'deployments_url': globals().get('deployments_url'), 'headers': globals().get('headers'), 'allowed_models': globals().get('allowed_models'), 'selected': globals().get('selected')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    record("discovery/list", transport.request("GET", deployments_url, headers=headers),
           body_check=lambda data: {item["name"] for item in data["value"]} == set(allowed_models))
    record("discovery/existing", transport.request("GET", f"{deployments_url}/{quote(selected['openai'], safe='')}",
                                                 headers=headers),
           body_check=lambda data: data.get("name") == selected["openai"])
    record("discovery/missing", transport.request("GET", f"{deployments_url}/nonexistent-validation-model-xyz",
                                                headers=headers), 404,
           body_check=lambda data: bool(error_code(data)))


## 6. OpenAI and inference patterns — real content plus routing headers

Exercise 6. OpenAI and inference patterns — real content plus routing headers and compare the actual response/evidence with the expected feature behavior.

**Input:** `openai_url`, `inference_url`, `selected`, `messages`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('6. OpenAI and inference patterns — real content plus routing headers', inputs={'openai_url': globals().get('openai_url'), 'inference_url': globals().get('inference_url'), 'selected': globals().get('selected'), 'messages': globals().get('messages')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    for case, url, model, payload, api_type in [
        ("openai/chat", openai_url, selected["openai"], {"messages": messages}, "openai"),
        ("inference/chat", inference_url, selected["inference"],
         {"model": selected["inference"], "messages": messages}, "inference"),
    ]:
        result = record(case, transport.request("POST", url, headers=headers, json=payload),
                        body_check=lambda data: bool(chat_content(data)), model=model)
        run.check(f"{case}/routing", result.headers.get("UAIG-API-Type") == api_type
                  and result.headers.get("UAIG-Model-Id") == model
                  and bool(result.headers.get("UAIG-Backend")) and bool(result.headers.get("UAIG-Final-Path")),
                  expected=f"{api_type} type, selected model, backend and final path",
                  actual={key: result.headers.get(key) for key in
                          ("UAIG-API-Type", "UAIG-Model-Id", "UAIG-Backend", "UAIG-Final-Path")})


## 7. Responses create/get/delete lifecycle

Select Responses capability independently of chat. Require actual ID, completed content, preserved identity and successful deletion.

Exercise 7. Responses create/get/delete lifecycle and compare the actual response/evidence with the expected feature behavior.

**Input:** `selected`, `test_responses_api`, `responses_url`, `headers`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('7. Responses create/get/delete lifecycle', inputs={'selected': globals().get('selected'), 'test_responses_api': globals().get('test_responses_api'), 'responses_url': globals().get('responses_url'), 'headers': globals().get('headers')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if "responses" in selected:
        result = record("responses/create",
                        transport.request("POST", responses_url, headers=headers,
                                          json={"model": selected["responses"], "input": "Say OK.", "store": True}),
                        body_check=lambda data: bool(data.get("id")) and data.get("status") == "completed"
                        and bool(data.get("output")))
        run.check("responses/routing", result.headers.get("UAIG-API-Type") == "responses",
                  expected="responses", actual=result.headers.get("UAIG-API-Type"))
        response_id = result.json_body.get("id") if isinstance(result.json_body, dict) else None
        if response_id:
            route = f"{responses_url}/{quote(response_id, safe='')}?model={quote(selected['responses'], safe='')}"
            retrieval = transport.request("GET", route, headers=headers)
            if retrieval.status_code == 404 and not retrieval.error:
                run.skip("responses/get", "Responses GET returned HTTP 404; retrieval not validated (warning).")
            else:
                record("responses/get", retrieval,
                       body_check=lambda data: data.get("id") == response_id and bool(data.get("output")))
            record("responses/delete", transport.request("DELETE", route, headers=headers),
                   body_check=lambda data: data.get("deleted") is True)
        else:
            run.blocked("responses/get-delete", "Creation did not return a response ID.")
    elif test_responses_api:
        run.skip("responses/lifecycle", "No eligible Responses model in this environment after exclusions.")
    else:
        run.skip("responses/lifecycle", "Responses coverage disabled explicitly.")


## 8. Optional Gemini OpenAI-compatible path

Exercise 8. Optional Gemini OpenAI-compatible path and compare the actual response/evidence with the expected feature behavior.

**Input:** `selected`, `test_gemini`, `gemini_url`, `headers`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8. Optional Gemini OpenAI-compatible path', inputs={'selected': globals().get('selected'), 'test_gemini': globals().get('test_gemini'), 'gemini_url': globals().get('gemini_url'), 'headers': globals().get('headers')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if "gemini" in selected:
        result = record("gemini/chat", transport.request("POST", gemini_url, headers=headers,
                        json={"model": selected["gemini"], "messages": messages}),
                        body_check=lambda data: bool(chat_content(data)))
        run.check("gemini/routing", result.headers.get("UAIG-API-Type") == "geminiopenai"
                  and result.headers.get("UAIG-Model-Id") == selected["gemini"],
                  expected="Gemini type and selected model", actual=result.headers.get("UAIG-API-Type"))
    elif test_gemini:
        run.skip("gemini/chat", "No eligible Gemini model in this environment after exclusions.")
    else:
        run.skip("gemini/chat", "Gemini coverage disabled explicitly.")


## 9. API key authentication and cross-product isolation

Missing/wrong keys must be rejected. A Unified-only contract must not authorize the separate OpenAI API.

Exercise 9. API key authentication and cross-product isolation and compare the actual response/evidence with the expected feature behavior.

**Input:** `openai_url`, `headers`, `result`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('9. API key authentication and cross-product isolation', inputs={'openai_url': globals().get('openai_url'), 'headers': globals().get('headers'), 'result': globals().get('result')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    record("auth/valid", transport.request("POST", openai_url, headers=headers, json={"messages": messages}),
           body_check=lambda data: bool(chat_content(data)))
    for case, auth in [("missing", {}), ("invalid", {"api-key": "invalid-validation-key-000000"})]:
        record(f"auth/{case}", transport.request("POST", openai_url, headers=auth, json={"messages": messages}), 401,
               body_check=lambda data: bool(data.get("message") or error_code(data)))
    try:
        other_api = apim.discover_api_exact("openai")
        result = transport.request(
            "POST", f"{other_api['url']}/deployments/{quote(selected['openai'], safe='')}/chat/completions?api-version={openai_api_version}",
            headers=headers, json={"messages": messages},
        )
        denial = result.json_body if isinstance(result.json_body, dict) else {}
        run.check("auth/cross-product", not result.error and result.status_code in (401, 403)
                  and bool(error_code(denial) or denial.get("message")),
                  expected="401/403 from separate API not granted to this product", actual=result.error or result.status_code)
    except Exception as exc:
        run.blocked("auth/cross-product", f"Separate OpenAI API is required for isolation coverage: {exc}")


## 10. Debug headers — presence and semantics, not print-success

Exercise 10. Debug headers — presence and semantics, not print-success and compare the actual response/evidence with the expected feature behavior.

**Input:** `openai_url`, `headers`, `messages`, `result`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('10. Debug headers — presence and semantics, not print-success', inputs={'openai_url': globals().get('openai_url'), 'headers': globals().get('headers'), 'messages': globals().get('messages'), 'result': globals().get('result')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    result = record("headers/chat", transport.request("POST", openai_url, headers=headers, json={"messages": messages}),
                    body_check=lambda data: bool(chat_content(data)))
    expected_headers = [
        "UAIG-Auth-Type", "UAIG-User-Id", "UAIG-Subscription", "UAIG-Model-Id", "UAIG-API-Type",
        "UAIG-Processed-Path", "UAIG-API-Version", "UAIG-Is-Streaming", "UAIG-Backend", "UAIG-Final-Path",
        "UAIG-Cache-Operation",
    ]
    run.check("headers/presence", all(result.headers.get(key) not in (None, "", "not-set") for key in expected_headers),
              expected=expected_headers, actual={key: result.headers.get(key) for key in expected_headers})
    run.check("headers/values", result.headers.get("UAIG-Is-Streaming") == "false"
              and result.headers.get("UAIG-API-Type") == "openai"
              and result.headers.get("UAIG-Cache-Operation") in ("cache-hit", "cache-miss"),
              expected="Nonstreaming OpenAI with valid cache operation", actual="observed debug values")


## 11. Blocked paths and model-level RBAC

A generic 403 or provider 400 is not proof; assert gateway policy error codes.

Exercise 11. Blocked paths and model-level RBAC and compare the actual response/evidence with the expected feature behavior.

**Input:** `inference_url`, `openai_url`, `headers`, `messages`, `base_url`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('11. Blocked paths and model-level RBAC', inputs={'inference_url': globals().get('inference_url'), 'openai_url': globals().get('openai_url'), 'headers': globals().get('headers'), 'messages': globals().get('messages'), 'base_url': globals().get('base_url')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    for index, path in enumerate(["invalid-path/chat/completions", "some/random/endpoint", "v2/openai/chat/completions"]):
        record(f"path-rejection/{index}", transport.request("POST", f"{base_url}/{path}", headers=headers,
               json={"model": selected["openai"], "messages": messages}), 403,
               body_check=lambda data: error_code(data) == "PathNotAllowed")
    record("rbac/unauthorized-model", transport.request("POST", inference_url, headers=headers,
           json={"model": "unauthorized-validation-model-xyz", "messages": messages}), 403,
           body_check=lambda data: error_code(data) == "unauthorized_model_access")
    record("rbac/allowed-control", transport.request("POST", openai_url, headers=headers, json={"messages": messages}),
           body_check=lambda data: bool(chat_content(data)))


## 12. Complete SSE streaming

Require event framing, valid JSON chunks, nonempty delta content, completion marker and true streaming header. No fabricated status on transport failure.

Exercise 12. Complete SSE streaming using the selected model and consume the visible stream without replacing it with a status-only check.

**Input:** `selected`, `base_url`, `openai_api_version`, `headers`, `messages`

**Output:** Stream chunks, terminal completion evidence, and the recorded nonempty-content/stream-completeness verdict.


In [ ]:
with utils.notebook_step('12. Complete SSE streaming', inputs={'selected': globals().get('selected'), 'base_url': globals().get('base_url'), 'openai_api_version': globals().get('openai_api_version'), 'headers': globals().get('headers'), 'messages': globals().get('messages')}, expected='Stream chunks, terminal completion evidence, and the recorded nonempty-content/stream-completeness verdict.', namespace=globals()):
    import requests
    streaming_url = (f"{base_url}/openai/deployments/{quote(selected['streaming'], safe='')}/chat/completions"
                     f"?api-version={openai_api_version}") if "streaming" in selected else None
    if not streaming_url:
        run.skip("streaming/complete", "No eligible streaming model after mapping and exclusions.")
    else:
      try:
        with utils.http_call(requests.post, streaming_url, headers=headers, json={"messages": messages, "stream": True},
                           timeout=60, stream=True) as response:
            chunks, parts, done, invalid = 0, [], False, False
            for line in response.iter_lines(decode_unicode=True):
                if isinstance(line, bytes):
                    line = line.decode("utf-8")
                if not line or line.startswith(":"):
                    continue
                if not line.startswith("data:"):
                    invalid = True
                    continue
                event = line[5:].strip()
                if event == "[DONE]":
                    done = True
                    break
                try:
                    chunk = json.loads(event)
                    chunks += 1
                    choices = chunk.get("choices") or []
                    if choices:
                        parts.append((choices[0].get("delta") or {}).get("content") or "")
                except (ValueError, TypeError, AttributeError):
                    invalid = True
            run.check("streaming/complete", response.status_code == 200 and "text/event-stream" in response.headers.get("Content-Type", "")
                      and response.headers.get("UAIG-Is-Streaming") == "true" and chunks > 0
                      and bool("".join(parts).strip()) and done and not invalid,
                      expected="Valid content-bearing SSE stream with DONE and streaming header",
                      actual=response.status_code, chunks=chunks, completion=done, text="".join(parts)[:200])
      except Exception as exc:
        run.check("streaming/complete", False, expected="Complete SSE stream", actual=type(exc).__name__)


## 13. Optional bounded stress sample

This is not feature-essential alert load; disabled by default. Every unexpected transport/body failure is retained.

Collect the final 13. Optional bounded stress sample verdict without discarding earlier failed or blocked required cases.

**Input:** `enable_stress`, `stress_requests`, `openai_url`, `headers`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('13. Optional bounded stress sample', inputs={'enable_stress': globals().get('enable_stress'), 'stress_requests': globals().get('stress_requests'), 'openai_url': globals().get('openai_url'), 'headers': globals().get('headers')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if enable_stress:
        if not 1 <= stress_requests <= 100:
            raise ValueError("stress_requests must be within 1..100.")
        for index in range(stress_requests):
            result = transport.request("POST", openai_url, headers=headers, json={"messages": messages})
            run.check(f"stress/{index}", not result.error and
                      (result.status_code == 429 or (result.status_code == 200 and bool(chat_content(result.json_body)))),
                      expected="Successful content or explicit rate limit", actual=result.error or result.status_code,
                      required=False, duration=result.duration)
    else:
        run.skip("stress", "Additional stress sample disabled explicitly.")
    run.finish()


## 14. Opt-in owned-resource cleanup

Apply the explicitly enabled 14. Opt-in owned-resource cleanup action only to resources owned by this run.

**Input:** `cleanup_enabled`, `product_id`, `artifact`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('14. Opt-in owned-resource cleanup', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'artifact': globals().get('artifact')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    utils.format_output('14. Opt-in owned-resource cleanup', {'value_1': "Owned cleanup candidates:", 'value_2': artifact.cleanup_candidates()}, status='info', part='RESULT')
    if cleanup_enabled:
        delete_owned_product(context, apim, product_id, artifact=artifact)
    else:
        utils.format_output('14. Opt-in owned-resource cleanup', {'value_1': "Cleanup disabled."}, status='info', part='RESULT')
